In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

In [ ]:
train = pd.read_csv('/content/sample_data/train_v2.csv', low_memory=False)
train.head()


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-01-30,5577,616,1,1,0,0
1,2,5,2015-01-30,5919,624,1,1,0,0
2,3,5,2015-01-30,6911,678,1,1,0,0
3,4,5,2015-01-30,13307,1632,1,1,0,0
4,5,5,2015-01-30,5640,617,1,1,0,0


In [ ]:
store = pd.read_csv('/content/sample_data/store.csv')
store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [ ]:
df = pd.merge(train, store, on='Store', how='left')
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-01-30,5577,616,1,1,0,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-01-30,5919,624,1,1,0,0,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-01-30,6911,678,1,1,0,0,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-01-30,13307,1632,1,1,0,0,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-01-30,5640,617,1,1,0,0,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [ ]:
print(df.isna().sum())


Store                            0
DayOfWeek                        0
Date                             0
Sales                            0
Customers                        0
Open                             0
Promo                            0
StateHoliday                     0
SchoolHoliday                    0
StoreType                        0
Assortment                       0
CompetitionDistance            162
CompetitionOpenSinceMonth    21312
CompetitionOpenSinceYear     21312
Promo2                           0
Promo2SinceWeek              34928
Promo2SinceYear              34928
PromoInterval                34928
dtype: int64


In [ ]:
print(df.isna().sum())

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
dtype: int64


In [ ]:
print(df.columns)


Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval'],
      dtype='object')


In [ ]:
categorical_cols_for_dummies = ['StateHoliday', 'StoreType', 'Assortment', 'PromoInterval','NoPromo']

# Check which of these columns actually exist in the DataFrame
existing_cols = [col for col in categorical_cols_for_dummies if col in df.columns]

# Apply get_dummies on the existing columns
df = pd.get_dummies(df, columns=existing_cols, drop_first=True)

# Convert any boolean columns created by get_dummies to 0/1
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
# Convert 'Date' column to datetime objects and extract features if 'Date' column exists
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])

    # Extract numerical features from 'Date'
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day

    # Drop the original 'Date' column as it's no longer needed
    df = df.drop('Date', axis=1)



In [ ]:
df.head()

,Store,DayOfWeek,Sales,Customers,Open,Promo,SchoolHoliday,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,...,StateHoliday_c,StoreType_b,StoreType_c,StoreType_d,Assortment_b,Assortment_c,PromoInterval_1,Year,Month,Day
0,1,5,5577,616,1,1,0,1270.0,9.0,2008.0,...,0,0,1,0,0,0,0,2015,1,30
1,2,5,5919,624,1,1,0,570.0,11.0,2007.0,...,0,0,0,0,0,0,1,2015,1,30
2,3,5,6911,678,1,1,0,14130.0,12.0,2006.0,...,0,0,0,0,0,0,1,2015,1,30
3,4,5,13307,1632,1,1,0,620.0,9.0,2009.0,...,0,0,1,0,0,1,0,2015,1,30
4,5,5,5640,617,1,1,0,29910.0,4.0,2015.0,...,0,0,0,0,0,0,0,2015,1,30


In [ ]:
# DayOfWeek
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 6 else 0)


In [ ]:
# How many years the competitor has been open
df['CompetitionOpenSinceYears'] = df['Year'] - df['CompetitionOpenSinceYear']
df['CompetitionOpenSinceYears'] = df['CompetitionOpenSinceYears'].apply(lambda x: 0 if x < 0 else x)

# Whether a competitor exists or not
df['CompetitorExists'] = df['CompetitionDistance'].apply(lambda x: 0 if x == 200000 else 1)


In [ ]:

X = df.drop('Sales', axis=1)
y = df['Sales']


In [ ]:
# Splitting dataset into test/train
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [ ]:
import numpy as np


y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)


Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

decisionTreeModel = DecisionTreeRegressor(
    criterion='squared_error',
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=10,
    random_state=1
)

decisionTreeModel.fit(X_train, y_train_log)


DecisionTreeRegressor(max_depth=5, min_samples_leaf=10, random_state=1)

In [ ]:
y_pred_log = decisionTreeModel.predict(X_test)

# convert back to the original (real) values
y_pred = np.expm1(y_pred_log)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.2f}")


MAE: 998.16
RMSE: 1524.98
R2: 0.87


In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

xgb_model.fit(X_train, y_train_log)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_pred_xgb_log = xgb_model.predict(X_test)
y_pred_xgb = np.expm1(y_pred_xgb_log)

In [ ]:
# MAE
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

# RMSE
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

# R2
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"MAE XGBoost: {mae_xgb:.2f}")
print(f"RMSE XGBoost: {rmse_xgb:.2f}")
print(f"R2 XGBoost: {r2_xgb:.4f}")

MAE XGBoost: 469.38
RMSE XGBoost: 704.17
R2 XGBoost: 0.9731
